<span style="font-family: 'Glass Antiqua', serif; font-size: 26px;">
Librerías
</span>

In [1]:
from pathlib import Path

import re
import numpy as np
import pandas as pd

<span style="font-family: 'Glass Antiqua', serif; font-size: 26px;">
Rutas de trabajo
</span>

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
FILE_ROUTE = INTERIM_DIR / "LIMA_METROPOLITANA_2026_08_CLEANED.xlsx"

<span style="font-family: 'Glass Antiqua', serif; font-size: 26px;">
DataFrame principal
</span>

In [4]:
df = pd.read_excel(FILE_ROUTE)

In [5]:
# GENERAL DATA SUMMARY
line = "=" * 70

print(f"\n{line}\nGENERAL DATA SUMMARY\n{line}")
print(f"Records          : {len(df):,}")
print(f"Columns          : {len(df.columns):,}")
print(f"Duplicate rows   : {df.duplicated().sum():,}")
print(f"Missing values   : {df.isna().sum().sum():,}")
print(f"Unique subtypes  : {df['subtipo'].nunique() if 'subtipo' in df else 'N/A'}")

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("\nMISSING VALUES BY COLUMN")
print("-" * 70)
print(missing.to_string() if not missing.empty else "No missing values.")
print(line)


GENERAL DATA SUMMARY
Records          : 8,410
Columns          : 137
Duplicate rows   : 0
Missing values   : 678
Unique subtypes  : 12

MISSING VALUES BY COLUMN
----------------------------------------------------------------------
fecha_venc    678


In [6]:
month_order = {
    month: i for i, month in enumerate([
        "ene", "feb", "mar", "abr", "may", "jun",
        "jul", "ago", "set", "oct", "nov", "dic",
    ])
}

pattern = re.compile(r"^(ene|feb|mar|abr|may|jun|jul|ago|set|oct|nov|dic)(\d{2})tot$", re.I)

total_columns = sorted(
    (col for col in df.columns if pattern.fullmatch(str(col))),
    key=lambda col: (
        int(pattern.fullmatch(str(col)).group(2)),
        month_order[pattern.fullmatch(str(col)).group(1).lower()],
    ),
    reverse=True,
)

if not total_columns:
    print("No monthly total columns found.")
else:
    results = []

    for col in total_columns:
        values = pd.to_numeric(df[col], errors="coerce")
        values = values.where(np.isfinite(values))
        valid = values.dropna()

        results.append({
            "Month": col[:-3],
            "Records": len(values),
            "Valid": len(valid),
            "Missing_or_invalid": int(values.isna().sum()),
            "Total": valid.sum(),
            "Mean": valid.mean(),
            "Median": valid.median(),
            "Std_dev": valid.std(),
            "Min": valid.min(),
            "Max": valid.max(),
            "Zeros": int(valid.eq(0).sum()),
        })

    summary = pd.DataFrame(results)

    line = "=" * 115
    print(f"\n{line}\nMONTHLY STATISTICS\n{line}")
    print(summary.to_string(index=False))
    print(line)


MONTHLY STATISTICS
Month  Records  Valid  Missing_or_invalid    Total        Mean  Median       Std_dev  Min      Max  Zeros
ago26     8410   8410                   0 40079572 4765.704162    99.0 184011.412735    0 16800996   1717
jul26     8410   8410                   0 40441250 4808.709869    96.0 186127.719142    0 16953629   1862
jun26     8410   8410                   0 43164164 5132.480856   100.0 209379.454760    0 19099434   1836
may26     8410   8410                   0 38958451 4632.396076    99.0 157756.800414    0 14328212   1833
abr26     8410   8410                   0 39846251 4737.960880    94.5 166401.518298    0 15011453   1817
mar26     8410   8410                   0 36946873 4393.207253    96.0 129377.689399    0 11584094   1866
feb26     8410   8410                   0 31963517 3800.656005    86.0  98578.582962    0  8772215   1922
ene26     8410   8410                   0 34363364 4086.012366    85.0 117161.037782    0 10421038   1871
dic25     8410   8410     

In [7]:
df["nombre_med"].value_counts()

nombre_med
LOSARTAN POTASICO 50 MG  TABLETA                               20
LORATADINA 10 MG  TABLETA                                      20
CLONAZEPAM 2 MG  TABLETA                                       20
PREDNISONA 50 MG  TABLETA                                      20
ESCOPOLAMINA N-BUTILBROMURO 10 MG  TABLETA                     20
                                                               ..
TOXINA BOTULINICA TIPO A 500 U  INYECTABLE                      1
SOLUCIONES PARA NUTRICION PARENTERAL 2 EN 1  2 L INYECTABLE     1
EMICIZUMAB 30 MG/ML 1ML INYECTABLE                              1
NITRATO DE PLATA 10 MG/ ML 15 ML SOLUCION                       1
DEXKETOPROFENO 25 MG  TABLETA                                   1
Name: count, Length: 1085, dtype: int64

In [9]:
df["uemef"].nunique()

18

In [10]:
df["establec"].nunique()

18

In [11]:
df["nombre_med"].nunique()

1085

In [12]:
resultado = (
    df.groupby("establec")["nombre_med"]
    .nunique()
    .reset_index(name="cantidad_nombre_med")
    .sort_values("cantidad_nombre_med", ascending=False)
)

print(resultado.to_string(index=False))

                                          establec  cantidad_nombre_med
                     HOSPITAL NACIONAL DOS DE MAYO                  678
                HOSPITAL NACIONAL CAYETANO HEREDIA                  650
                HOSPITAL NACIONAL ARZOBISPO LOAYZA                  615
                        HOSPITAL MARIA AUXILIADORA                  591
                 HOSPITAL NACIONAL HIPOLITO UNANUE                  583
                      HOSPITAL DE APOYO SANTA ROSA                  582
         HOSPITAL DE EMERGENCIAS VILLA EL SALVADOR                  579
              HOSPITAL NACIONAL SERGIO E. BERNALES                  510
                   HOSPITAL SAN JUAN DE LURIGANCHO                  503
                  HOSPITAL CARLOS LANFRANCO LA HOZ                  433
HOSPITAL NACIONAL DOCENTE MADRE NIÑO SAN BARTOLOME                  433
                    HOSPITAL DE LIMA ESTE -VITARTE                  423
 HOSPITAL DE MEDIANA COMPLEJIDAD JOSE AGURTO TELLO              

In [5]:
df["indicador"].value_counts(dropna=False)

indicador
SOBRESTOCK              3739
NORMOSTOCK              2333
SUBSTOCK                1039
DESABASTECIDO            432
SIN CONSUMO              328
SIN ROTACION             306
NO APLICA                122
NO APLICA DONACIONES      83
SIN INFORMACION           28
Name: count, dtype: int64